1. Importing Libraries

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
from glob import glob
import cv2

from keras.models import Model, Sequential
from keras.layers import Dense, Flatten, Conv2D, MaxPooling2D
#from keras.layers.normalization import BatchNormalization
import os
import seaborn as sns
from keras.applications.densenet import DenseNet169,DenseNet121,DenseNet201
from keras.applications.nasnet import NASNetMobile,NASNetLarge
from keras.applications.xception import Xception
from keras.applications.resnet import ResNet101, ResNet50, ResNet152
from keras.applications.resnet_v2 import ResNet50V2,ResNet101V2,ResNet152V2
from keras.applications.mobilenet_v2 import MobileNetV2
from keras.applications.mobilenet import MobileNet
from keras.applications.inception_resnet_v2 import InceptionResNetV2
from keras.applications.inception_v3 import InceptionV3
from keras.applications.vgg19 import VGG19
from keras.applications.vgg16 import VGG16
from keras.applications.efficientnet import EfficientNetB0

2. Mounting Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

3. Dataset Loading and Class Definition

In [ ]:
images_path = {}
images_path["Covid"] = glob("/content/drive/MyDrive/Dataset/COVID-19-master/X-Ray Image DataSet/Covid-19/*.png")
images_path["Covid"] += glob("/content/drive/MyDrive/Dataset/COVID-19-master/X-Ray Image DataSet/Covid-19/*.jpeg")
images_path["Covid"] += glob("/content/drive/MyDrive/Dataset/COVID-19-master/X-Ray Image DataSet/Covid-19/*.jpg")

images_path["NoFindings"] = glob("/content/drive/MyDrive/Dataset/COVID-19-master/X-Ray Image DataSet/No_findings/*.png")
images_path["NoFindings"] += glob("/content/drive/MyDrive/Dataset/COVID-19-master/X-Ray Image DataSet/No_findings/*.jpeg")
images_path["NoFindings"] += glob("/content/drive/MyDrive/Dataset/COVID-19-master/X-Ray Image DataSet/No_findings/*.jpg")

print(images_path["Covid"])
print(images_path["NoFindings"])

images_class = {
    "Covid": 0,
    "NoFindings": 1
}

In [ ]:
SIZE = [224,224]
train_images = []
train_labels  = []

4. Image Preprocessing and Label Assignment

In [ ]:
for label in images_path:
    for image_path in images_path[label]:
        image = cv2.imread(image_path)
        image = cv2.resize(image,(224, 224))
        train_images.append(image)
        train_labels.append(images_class[label])

In [ ]:
images = np.array(train_images)
labels = np.array(train_labels)

5. Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder
label = LabelEncoder()
labels = label.fit_transform(labels)
labels.shape

In [ ]:
images = images/255.0

6. Pre-trained CNN Architecture Selection

In [ ]:
SIZE = 224
model = DenseNet169(include_top=False, input_shape=(224, 224, 3), pooling='avg')
#model =DenseNet121(include_top=False, input_shape=(SIZE, SIZE, 3), pooling="avg")
#model = DenseNet201(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = VGG16(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = VGG19(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = InceptionV3(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = MobileNet(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = MobileNetV2(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = ResNet101(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = ResNet50(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = ResNet50V2(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = ResNet101V2(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = ResNet152V2(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = InceptionResNetV2(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = ResNet152(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = Xception(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = NASNetMobile(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')
#model = EfficientNetB0(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')

6.1. Freezing the Pre-trained Model Layers

In [ ]:
for layer in model.layers:
    layer.trainable = False

6.2. Deep Feature Extraction

In [ ]:
features = model.predict(images)
features = features.reshape(features.shape[0],-1)

7. Feature Selection Using Chi-Square

In [ ]:
from sklearn.feature_selection import f_classif,chi2
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import SelectKBest

def select_features(X_train, y_train, X_test,i):
    # configure to select all features
    #fs = SelectKBest(score_func=f_classif, k=i)
    fs = SelectKBest(score_func=chi2, k=i)
    # learn relationship from training data
    fs.fit(X_train, y_train)
    # transform train input data
    X_train_fs = fs.transform(X_train)
    # transform test input data
    X_test_fs = fs.transform(X_test)
    return X_train_fs, X_test_fs

8. Classification and Performance Evaluation Libraries

In [ ]:
from sklearn.metrics import f1_score
import statistics
from sklearn.metrics import confusion_matrix,accuracy_score
from xgboost import XGBClassifier
from sklearn.metrics import precision_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import statistics
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_validate

9. Binary Classification Using XGBoost

In [ ]:

av = 0
place=0

cv = KFold(n_splits=5, shuffle=True)
f = 0
result = dict()
result["Accuracy score"] = []
f1 = []
pre_score = []
sensitivity = []
specivity = []

for train_ix, test_ix in cv.split(features):
    # get data
    train_X, test_X = features[train_ix], features[test_ix]
    train_y, test_y = labels[train_ix], labels[test_ix]
    train_X,test_X = select_features(train_X,train_y,test_X,1000)
    # fit model
    model = XGBClassifier(learning_rate=0.50,n_estimators=100,seed=0)
    model = XGBClassifier()
    model.fit(train_X, train_y)
    # evaluate model
    yhat = model.predict(test_X)
    acc = accuracy_score(test_y, yhat)
    print(f"\n confusion matrix fold {f}")
    f = f+1
    print(confusion_matrix(test_y, yhat))
    Y_test = test_y
    Y_pred = yhat
    result["Accuracy score"].append(accuracy_score(Y_test, Y_pred))
    result["confusion matrix"] = confusion_matrix(Y_test, Y_pred)
    result["f1 score"] = f1_score(Y_test, Y_pred, average="macro")
    result["precision score"] = precision_score(Y_test, Y_pred, average="macro")
    result["Sensitivity"] = result["confusion matrix"][0,0] / (result["confusion matrix"][0,0] + result["confusion matrix"][0,1])
    result["Specifity"] = result["confusion matrix"][1,1] / (result["confusion matrix"][1,0] + result["confusion matrix"][1,1])
    f1.append(f1_score(Y_test, Y_pred, average="macro"))
    pre_score.append(precision_score(Y_test, Y_pred, average="macro"))
    sensitivity.append(result["confusion matrix"][0,0] / (result["confusion matrix"][0,0] + result["confusion matrix"][0,1]))
    specivity.append(result["confusion matrix"][1,1] / (result["confusion matrix"][1,0] + result["confusion matrix"][1,1]))
    
    
    print(f"\nAccuracy score : ", result["Accuracy score"])
    print(f"\nconfusion matrix : \n", result["confusion matrix"])
    print(f"\nf1 score : ", result["f1 score"])
    print(f"\nprecision score : ", result["precision score"])
    print(f"\nSpecifity : ", result["Specifity"])
    print(f"\nSensitivity : ", result["Sensitivity"])
    print(f"\n============")

print(f"\nAccuracy score : ", statistics.mean(result["Accuracy score"]))    
print(f"\nf1 score : ", statistics.mean(f1))
print(f"\nprecision score : ", statistics.mean(pre_score))
print(f"\nSpecifity : ", statistics.mean(specivity))
print(f"\nSensitivity : ", statistics.mean(sensitivity))

10. Cross-Validation and Confusion Matrix Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
import seaborn as sn
import pandas as pd
import matplotlib.pyplot as plt
for i in range (200):
  l =[]
  result["Accuracy score All"] = []
  overlapped = np.array([[0,0],[0,0]])
  class_names=['Covid-19','No_findings']
  f = 0
  for train_ix, test_ix in cv.split(features):
      # get data
      train_X, test_X = features[train_ix], features[test_ix]
      train_y, test_y = labels[train_ix], labels[test_ix]
      train_X,test_X = select_features(train_X,train_y,test_X,1000)
      # fit model
      model = XGBClassifier(learning_rate=0.50,n_estimators=100,seed=0)
      model.fit(train_X, train_y)
      # evaluate model
      yhat = model.predict(test_X)
      acc = accuracy_score(test_y, yhat)
      Y_test = test_y
      Y_pred = yhat
      result["Accuracy score All"].append(accuracy_score(Y_test, Y_pred))
      result["confusion matrix"] = confusion_matrix(Y_test, Y_pred)
      result["f1 score"] = f1_score(Y_test, Y_pred, average="macro")
      result["precision score"] = precision_score(Y_test, Y_pred, average="macro")
      result["Sensitivity"] = result["confusion matrix"][0,0] / (result["confusion matrix"][0,0] + result["confusion matrix"][0,1])
      result["Specifity"] = result["confusion matrix"][1,1] / (result["confusion matrix"][1,0] + result["confusion matrix"][1,1])
      f1.append(f1_score(Y_test, Y_pred, average="macro"))
      pre_score.append(precision_score(Y_test, Y_pred, average="macro"))
      sensitivity.append(result["confusion matrix"][0,0] / (result["confusion matrix"][0,0] + result["confusion matrix"][0,1]))
      specivity.append(result["confusion matrix"][1,1] / (result["confusion matrix"][1,0] + result["confusion matrix"][1,1]))
      print(f"\nAccuracy score : ", result["Accuracy score All"])
      print(f"\nconfusion matrix : \n", result["confusion matrix"])
      print(f"\nf1 score : ", result["f1 score"])
      print(f"\nprecision score : ", result["precision score"])
      print(f"\nSpecifity : ", result["Specifity"])
      print(f"\nSensitivity : ", result["Sensitivity"])
      print(f"\n============")
      f = f+1
      l = confusion_matrix(Y_test, Y_pred)
      for i in range(len(l[0])):
          for j in range(len(l[0])):
              overlapped[i][j] = l[i][j] + overlapped[i][j]
      group_count = ['{0:0.0f}'.format(value) for value in
                  l.flatten()]
      group_counts=[]
      group_percentages = []
      for i in group_count:
          group_counts.append(f'({i})')
      for i in range(0,2):
          for j in range(0,2):
              group_percentages.append(l[i,j]/np.sum(l[:,j]))
            
      group_percentages = ['{0:.2%}'.format(value) for value in
                  group_percentages]        
      d = [f'{v2}\n{v3}' for  v2, v3 in
                zip(group_percentages,group_counts)]                     
      d = np.asarray(d).reshape(2,2)                    
      df_cm = pd.DataFrame(l, index = [i for i in class_names],
                        columns = [i for i in class_names])
      plt.figure(figsize = (6,4))
      ax = plt.axes()
      sns = sn.heatmap(df_cm, annot=d,ax = ax,cbar=False,fmt="",cmap="Blues")
      ax.set_title(f'Fold {f}',fontsize = 17)
      plt.show()
      figure = sns.get_figure()    
      figure.savefig(f'Fold{f}.svg')
      plt.clf()
  group_count = ['{0:0.0f}'.format(value) for value in
              overlapped.flatten()]
  group_counts=[]
  group_percentages = []
  for i in group_count:
      group_counts.append(f'({i})')
  for i in range(0,2):
      for j in range(0,2):
          group_percentages.append(overlapped[i,j]/np.sum(overlapped[:,j]))
  group_percentages = ['{0:.2%}'.format(value) for value in
              group_percentages]        
    
  d = [f'{v2}\n{v3}' for  v2, v3 in
            zip(group_percentages,group_counts)]                     

  d = np.asarray(d).reshape(2,2)                    

  df_cm = pd.DataFrame(overlapped, index = [i for i in class_names],
                    columns = [i for i in class_names])
  plt.figure(figsize = (6,4))
  ax = plt.axes()
  sns = sn.heatmap(df_cm, annot=d,ax = ax,cbar=False,fmt="",cmap="Blues")
  ax.set_title('Overlapped',fontsize = 17)
  plt.show()
  figure = sns.get_figure()    
  figure.savefig(f'Overlapped.svg')
  plt.clf()
  print(f"\nAccuracy score : ", statistics.mean(result["Accuracy score All"]))    
  print(f"\nf1 score : ", statistics.mean(f1))
  print(f"\nprecision score : ", statistics.mean(pre_score))
  print(f"\nSpecifity : ", statistics.mean(specivity))
  print(f"\nSensitivity : ", statistics.mean(sensitivity))  